<a href="https://colab.research.google.com/github/vlademirribeiro/3386-git-github-projeto_inicial/blob/master/churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

#importação das bibliotecas
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [ ]:

# 1. CARGA DE DADOS (Dataset Oficial IBM)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)


In [ ]:
df.head()

In [ ]:
df.info()


In [ ]:

pd.options.display.max_columns = None
df.head()

In [ ]:
# 2. PRÉ-PROCESSAMENTO (Limpeza Essencial)
# Converter TotalCharges para numérico (tratar erros de string vazia)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

In [ ]:
df.info()


In [ ]:
df.shape

In [ ]:
# Remover ID (não preditivo)
df_clean = df.drop(columns=['customerID'])

In [ ]:
le = LabelEncoder()
for coluna in df_clean.columns:
    if df_clean[coluna].dtype == 'object':
        df_clean[coluna] = le.fit_transform(df_clean[coluna])

In [ ]:
# 3. SEPARAÇÃO DE DADOS (Treino vs Teste)
X = df_clean.drop(columns=['Churn']) # Features
y = df_clean['Churn']                # Target (O que queremos prever)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 4. TREINAMENTO DO MODELO
print("⚡ Iniciando treinamento do modelo...")
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)


In [ ]:
# 5. VALIDAÇÃO DE PERFORMANCE
previsoes = modelo.predict(X_test)
acuracia = accuracy_score(y_test, previsoes)

In [ ]:
# Opcional: Visualização Simples
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, previsoes), annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusão")
plt.show()

In [ ]:
print("🌲 Treinando Random Forest...")
modelo_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42)



In [ ]:
modelo_rf.fit(X_train, y_train)

prev_rf = modelo_rf.predict(X_test)
acc_rf = accuracy_score(y_test, prev_rf)



In [ ]:
# MODELO XGBoost
print("🚀 Treinando XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)


In [ ]:
# PREVISÃO E ACURÁCIA
xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)




In [ ]:
print("="*40)
print(f"🏆 ACURÁCIA BASE: {acuracia:.2%}")
print("="*40)

print("="*40)
print(f"🏆 ACURÁCIA RANDOM FOREST: {acc_rf:.2%}")
print("="*40)

print("="*40)
print(f"🔥 ACURÁCIA XGBoost: {xgb_acc:.2%}")
print("="*40)



In [ ]:
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, prev_rf), annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusão - Random Forest")
plt.show()

In [ ]:

cm_xgb = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusão - XGBoost")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.tight_layout()
plt.show()


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


In [ ]:
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

# 1) Train-test split (sem SMOTE aqui)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from imblearn.over_sampling import SMOTE

# 2) Aplicar SMOTE SOMENTE no treino
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Antes SMOTE:", y_train.value_counts())
print("Depois SMOTE:", y_train_sm.value_counts())


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Random Forest com SMOTE
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"Acurácia RF com SMOTE: {rf_acc:.2%}")

# XGBoost com SMOTE
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_sm, y_train_sm)
xgb_pred = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)

print(f"Acurácia XGBoost com SMOTE: {xgb_acc:.2%}")


In [ ]:
from imblearn.combine import SMOTEENN
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


In [ ]:
# 1) Train-test split mantendo proporção
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2) SMOTE + ENN SOMENTE no treino
smote_enn = SMOTEENN(random_state=42)
X_train_se, y_train_se = smote_enn.fit_resample(X_train, y_train)

print("Antes:", y_train.value_counts())
print("Depois SMOTEENN:", y_train_se.value_counts())


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_se, y_train_se)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"Acurácia RF (SMOTEENN): {rf_acc:.2%}")

# XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_se, y_train_se)
xgb_pred = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f"Acurácia XGBoost (SMOTEENN): {xgb_acc:.2%}")


In [ ]:
# Depois de treinar o modelo
y_pred = modelo.predict(X_test)


In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label=1)  # 1 = classe churn

print(f"Acurácia : {acc:.2%}")
print(f"Precisão : {prec:.2%}")


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)  # ou X_train_sm / X_train_se


In [ ]:

from sklearn.metrics import accuracy_score, precision_score

modelos = [
    ("Logistic", log_model),
    ("Random Forest", rf_model),
    ("XGBoost", xgb_model),
]

for nome, modelo in modelos:
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1)

    print("=" * 40)
    print(f"Modelo   : {nome}")
    print(f"Acurácia : {acc:.2%}")
    print(f"Precisão : {prec:.2%}")
    print("=" * 40)
